<a href="https://colab.research.google.com/github/marianagarzaa/4010/blob/main/02_simulator_baseline.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Simulator and Baseline

## installs

In [2]:
!pip install -q 'lerobot[dataset]'
!pip install -q "mujoco==3.8.1" dm-env dm-tree glfw lxml pyopengl pyparsing absl-py protobuf scipy "imageio[ffmpeg]"
!pip install -q "dm-control==1.0.40" gym-aloha --no-deps

In [6]:
!python -m lerobot.processor.migrate_policy_normalization \
    --pretrained-path lerobot/act_aloha_sim_transfer_cube_human \
    --output-dir models/act_pretrained_migrated

Loading model from lerobot/act_aloha_sim_transfer_cube_human...

model.safetensors: downloading bytes:  67% 139M/207M [00:01<00:00, 222MB/s, 9.22MB/s  ] 
model.safetensors: reconstructing file:  32% 67.1M/207M [00:01<00:02, 54.4MB/s]
model.safetensors: downloading bytes:  85% 175M/207M [00:01<00:00, 202MB/s, 15.9MB/s  ]
model.safetensors: downloading bytes: 100% 194M/194M [00:01<00:00, 120MB/s, 18.1MB/s  ]
model.safetensors: reconstructing file: 100% 207M/207M [00:01<00:00, 128MB/s, 19.6MB/s  ]
config.json: 100% 1.33k/1.33k [00:00<00:00, 743kB/s]
train_config.json: 100% 5.29k/5.29k [00:00<00:00, 8.63MB/s]
Extracting normalization statistics...
Found normalization statistics for: ['observation.images.top', 'observation.state', 'action']
Detecting features and normalization modes...
Found normalization_mapping in config: {'VISUAL': 'MEAN_STD', 'STATE': 'MEAN_STD', 'ACTION': 'MEAN_STD'}
Detected features: ['observation.images.top', 'observation.state', 'action']
Normalization modes: {<Fea

##setup

In [3]:
import os
os.environ["MUJOCO_GL"] = "egl"

from google.colab import userdata
os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")

import gymnasium as gym
import gym_aloha

In [4]:
env = gym.make("gym_aloha/AlohaTransferCube-v0", obs_type="pixels_agent_pos")
obs, info = env.reset(seed=0)

print(obs["pixels"]["top"].shape, obs["agent_pos"].shape)
print(env.action_space)

(480, 640, 3) (14,)
Box(-1.0, 1.0, (14,), float32)


In [5]:
import imageio
from IPython.display import Video

obs, info = env.reset(seed=0)
frames = []

for step in range(400):
    action = env.action_space.sample()      # a random command
    obs, reward, terminated, truncated, info = env.step(action)
    frames.append(obs["pixels"]["top"])
    if terminated or truncated:
        break

print("success?", info["is_success"], "| steps:", step + 1)
imageio.mimsave("random_policy.mp4", frames, fps=50)
Video("random_policy.mp4", embed=True, width=480)

success? False | steps: 300


In [8]:
import time
start = time.time()

!MUJOCO_GL=egl lerobot-eval \
    --policy.path=models/act_pretrained_migrated \
    --env.type=aloha \
    --env.task=AlohaTransferCube-v0 \
    --eval.n_episodes=10 \
    --eval.batch_size=5 \
    --policy.device=cuda \
    --output_dir=outputs/eval/pretrained

print(f"Took {(time.time() - start) / 60:.1f} minutes")

WARNING 2026-11-19 23:06:14 figs/eval.py:73 No job name provided, using 'aloha_act' as job name.
INFO 2026-11-19 23:06:14 bot_eval.py:813 {'env': {'disable_env_checker': True,
         'episode_length': 400,
         'features': {},
         'features_map': {},
         'fps': 50,
         'max_parallel_tasks': 1,
         'obs_type': 'pixels_agent_pos',
         'render_mode': 'rgb_array',
         'server': None,
         'task': 'AlohaTransferCube-v0'},
 'eval': {'batch_size': 5,
          'max_episode_steps': None,
          'n_episodes': 10,
          'recording': False,
          'recording_private': False,
          'recording_repo_id': None,
          'use_async_envs': True},
 'job_name': 'aloha_act',
 'output_dir': PosixPath('outputs/eval/pretrained'),
 'policy': {'chunk_size': 100,
            'device': 'cuda',
            'dim_feedforward': 3200,
            'dim_model': 512,
            'dropout': 0.1,
            'dtype': None,
            'feedforward_activation': 'relu',

In [13]:
from IPython.display import Video
base = "outputs/eval/pretrained/videos/AlohaTransferCube-v0_0/"
Video(base + "eval_episode_2.mp4", embed=True, width=480)